In [29]:
# import re
import json
# import html 
import requests
import xml.etree.ElementTree as ET 
# from bs4 import BeautifulSoup

In [30]:
# Airport Weather Reporter 
# You work for a small startup 
# that displays up-to-date METAR weather reports for airports. 

# to fetch and process the METAR report from the Aviation Weather API 
# for any airport. 

# For example, you might choose Kansas City International Airport (station code KMCI). 
# Reference: https://aviationweather.gov/data/api/#api for detailed API specification. 

# 1. Make an HTTP GET request to 
# https://aviationweather.gov/api/data/metar?ids=KMCI&format=xml 
def fetch_metar(station_id: str) -> str:
        # [Fail] url = 'https://aviationweather.gov/api/data/metar?ids=KMCI&format=xml'
        # Reason: URL에 KMCI가 고정되어 있어 station_id = 'KJFK'가 다른공항을 처리하지 못함
        # Above url can't be used due to fixed KMCI, like 'station_id = 'KFJK' 
        url = 'https://aviationweather.gov/api/data/metar' 
        # I have to use this: if we change 'station_id', URL wouldn't change
        
        params = {
                # [Fail] "stations_id": stations_id
                # API는 반드시!!! "ids" 키를 요구함 함부로 바꾸면 안 됨
                # I have use the accurate query key name, DO NOT CHANGE (e.g. KMCI)
                "ids": station_id,  
                "format": "xml"
        }
        
        # [Class] requests.get(URL, params=params, timeout=10)
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:  
                print(f"Success getting XML response ({len(response.text)})")
                return response.text
        else:
                print(f"Fail due to error {response.status_code}")
                return None

# 2. Parse the XML response, locate the <METAR> tag, 
# and extract the following fields 
def parse_xml(xml_str: str):
    try: 
        root = ET.fromstring(xml_str)
        # [Fail] metar = root.find()
        # Reason: find()는 태그할 경로 필요함. "METAR" 경로를 반드시 호출하기 (시간부족)
        # 전체 tree에서 METAR를 찾도록 경로 추가하기 
        metar = root.find(".//METAR")
        
        # [Fail]: 이 함수에서 반드시 root와 metar을 반환해야함
        # parse_xml() 함수가 None을 반환하면 extract 함수에서 사용하지 못하므로 명시해주기 
        # I have to make if() to check 'metar' returns 'None'
        if metar is not None:
                return root, metar 
        else:
                print("No METAR element found")
                # root는 반환하고, metar는 None 출력 
                return root, None
        
    except ET.ParseError as e:
        print("XML parse error:", e)
        return None, None 

# (elements) from that tag: 
        # o station_id 
        # o latitude 
        # o longitude 
        # o temp_c 
        # o dewpoint_c 
        # o wind_dir_degrees 
        # o altim_in_hg 
        # o wind_speed_kt 
# 의미: <METAR> tag에서 위 elements(field)를 extract 하라는 것 
# meaning: I have to extract above elements from that tag(==METAR)

""" [FAIL!!!!!]
        station_id = elem.find("station_id")
        latitude = elem.find("latitude") 
        longitude = elem.find("longitude")
        temp_c = elem.find("temp_c")
        dewpoint_c = elem.find("dewpoint_c")
        wind_dir_degrees = elem.find("wind_dir_degrees")
        altim_in_hg = elem.find("altim_ing_hg")
        wind_speed_kt = elem.find("wind_spped_kt")
        if station_id is None or latitude is None o
""" 
# Reason:
# 1. 태그명 사용할 때 ctrl-c, ctrl-v 하기 (오타 주의)
# 2. elem.find("station_id")는 문자열이기 때문에 int() 변환 실수 
# 3. dewpoint_c.post 틀린 이유: XML Element에는 .post라는 속성이 없음 
# 기존 .text 또는 findtext() 이런 것만 사용해야함  

def extract_elements_from_metar(metar: ET.Element) -> dict:
        if metar is None:
                return {} 
        
        # (Class) elem.findtext("tag")처럼 추출하기
        data = {
                # String
                "station_id":metar.findtext("station_id", "N/A"),
                
                # Float
                "latitude": float(metar.findtext("latitude") or 0),
                "longitude": float(metar.findtext("longitude") or 0),
                "temp_c": float(metar.findtext("temp_c") or 0),
                "dewpoint_c": float(metar.findtext("dewpoint_c") or 0),
                "wind_dir_degrees": float(metar.findtext("wind_dir_degrees") or 0),
                "altim_in_hg": float(metar.findtext("altim_in_hg") or 0),
                "wind_speed_kt": float(metar.findtext("wind_speed_kt") or 0),
        }
        return data

# 3. Convert the extracted data into JSON format (e.g., a JSON object/dictionary with those 
# key-value pairs). 

def metar_to_json(weather_dict: dict) -> str:
        # (Class) json.dumps로 dictionary를 json 문자열로 반환했음
        return json.dumps(weather_dict, indent = 2)


# 4. Using those extracted values, 
# create a Python class called WeatherReport with attributes 
# for those fields (you decide the types, e.g., strings or floats). 
# Instantiate a WeatherReport object using the data you retrieved. 
class WeatherReport: 
        # [Fail]: class 속성 정의와 instance를 빼먹음 (초반내용)
        def __init__(self, station_id: str, 
                     latitude: float, longitude: float,
                     temp_c: float, dewpoint_c: float,
                     wind_dir_degrees: float, altim_in_hg: float,
                     wind_speed_kt: float):
                
                self.station_id = station_id
                self.latitude = latitude
                self.longitude = longitude
                self.temp_c = temp_c
                self.dewpoint_c = dewpoint_c
                self.wind_dir_degrees = wind_dir_degrees
                self.altim_in_hg = altim_in_hg
                self.wind_speed_kt = wind_speed_kt
                

# MAIN 

if __name__ == "__main__":
        print("Airport Weather Reporter\n")
        print("------------------------\n")
        
        # 1 
        # METAR 요청하기 
        # HTTP GET request 
        xml_data = fetch_metar("KMCI")
        
        if xml_data:
                root, metar = parse_xml(xml_data)
                # Parse XML과 <METAR> 위치 확인 
                
                # METAR 태그가 존재하는 경우만 데이터 처리
                if metar is not None:
                        # 2
                        # extract weater elements(fields) from METAR tag
                        # XML tag에서 8개 elements를 추출해서 Dict로 저장 
                        weather_data = extract_elements_from_metar(metar)
                        print("Extracted elements from metar: ", weather_data)
                        print("\n")
                        
                        # 3
                        # convert 'dict' to 'JSON' 
                        json_str = metar_to_json(weather_data)
                        print("Json: ", json_str)
                        print("\n")
                        
                        # 4
                        # 추출한 값들을 전달 
                        all_weather_data = WeatherReport(
                                weather_data["station_id"],
                                weather_data["latitude"],
                                weather_data["longitude"],
                                weather_data["temp_c"],
                                weather_data["dewpoint_c"],
                                weather_data["wind_dir_degrees"],
                                weather_data["altim_in_hg"],
                                weather_data["wind_speed_kt"]
                        )
                        
                        # ° # print object attribute 
                        # 생성된 object 객체의 attribute 속성을 사용하여 최종결과 출력 
                        print(f"Weather Report: {all_weather_data.station_id}, Temp: {all_weather_data.temp_c}°C")
                else:
                        # 만약 METAR tag 못 찾으면 예외처리 해주기 
                        print("Fail: Parse METAR")
        else:
                # API 호출자체를 실패할 경우 예외처리 해주기 
                print("Fail: fetch data")


Airport Weather Reporter

------------------------

Success getting XML response (1307)
Extracted elements from metar:  {'station_id': 'KMCI', 'latitude': 39.2975, 'longitude': -94.7309, 'temp_c': 16.1, 'dewpoint_c': -5.6, 'wind_dir_degrees': 0.0, 'altim_in_hg': 30.1, 'wind_speed_kt': 0.0}


Json:  {
  "station_id": "KMCI",
  "latitude": 39.2975,
  "longitude": -94.7309,
  "temp_c": 16.1,
  "dewpoint_c": -5.6,
  "wind_dir_degrees": 0.0,
  "altim_in_hg": 30.1,
  "wind_speed_kt": 0.0
}


Weather Report: KMCI, Temp: 16.1°C
